Set up packages, Colab paths, data folders, and result folders.


In [1]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TQDM_DISABLE"] = "1"


def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


for import_name, pip_name in [
    ("thop", "thop"),
    ("torchinfo", "torchinfo"),
    ("transformers", "transformers"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("tqdm", "tqdm"),
]:
    ensure_package(import_name, pip_name)

try:
    from huggingface_hub.utils import disable_progress_bars

    disable_progress_bars()
except Exception:
    pass


def project_paths(problem_name):
    here = Path.cwd()
    drive_root = Path("/content/drive/MyDrive")

    if Path("/content").exists() and not drive_root.exists():
        try:
            from google.colab import drive

            drive.mount("/content/drive")
        except Exception:
            pass

    if drive_root.exists():
        work_dir = drive_root / "homework_5"
    elif Path("/content").exists():
        work_dir = Path("/content/homework_5")
    else:
        work_dir = here / "homework_5"

    result_dir = work_dir / "results" / problem_name
    data_dir = work_dir / "data"
    result_dir.mkdir(parents=True, exist_ok=True)
    data_dir.mkdir(parents=True, exist_ok=True)
    return work_dir, data_dir, result_dir


Define shared helpers for reproducibility, device selection, CIFAR-100 loading, metrics, exports, plots, and training loops.


In [2]:
"""Reproducibility and runtime-device helpers."""


import os
import random
from dataclasses import dataclass

import numpy as np
import torch


@dataclass(frozen=True)
class DeviceInfo:
    """Small serializable description of the selected torch device."""

    device: torch.device
    name: str
    cuda_available: bool
    cuda_device_count: int

    def as_dict(self) -> dict[str, object]:
        return {
            "device": str(self.device),
            "name": self.name,
            "cuda_available": self.cuda_available,
            "cuda_device_count": self.cuda_device_count,
        }


def is_colab() -> bool:
    """Return True when running inside a Google Colab runtime."""

    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in os.environ


def set_seed(seed: int = 42, deterministic: bool = False) -> None:
    """Set practical random seeds for Python, NumPy, and PyTorch."""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = True


def select_device(prefer_cuda: bool = True) -> DeviceInfo:
    """Select CUDA when available, otherwise CPU."""

    cuda_available = bool(torch.cuda.is_available())
    if prefer_cuda and cuda_available:
        device = torch.device("cuda")
        name = torch.cuda.get_device_name(device)
    else:
        device = torch.device("cpu")
        name = "CPU"
    return DeviceInfo(
        device=device,
        name=name,
        cuda_available=cuda_available,
        cuda_device_count=torch.cuda.device_count(),
    )


def cuda_synchronize_if_needed(device: torch.device | str) -> None:
    """Synchronize CUDA timings only when the active device is CUDA."""

    device = torch.device(device)
    if device.type == "cuda":
        torch.cuda.synchronize(device)


"""Metrics, complexity estimates, plotting, and structured exports."""


import csv
import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import torch


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def count_parameters(model: torch.nn.Module) -> tuple[int, int]:
    """Return total and trainable parameter counts."""

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def accuracy_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    """Compute top-1 accuracy for a batch of raw logits."""

    predictions = logits.argmax(dim=1)
    return (predictions == labels).float().mean().item()


def model_size_megabytes(model: torch.nn.Module) -> float:
    """Estimate parameter storage size in MiB."""

    bytes_total = sum(p.numel() * p.element_size() for p in model.parameters())
    return bytes_total / (1024**2)


def estimate_macs_thop(
    model: torch.nn.Module,
    input_size: tuple[int, int, int, int],
    device: torch.device | str,
) -> dict[str, Any]:
    """Estimate MACs with thop and label the convention explicitly."""

    try:
        from thop import profile
    except Exception as exc:  # pragma: no cover - optional dependency
        return {
            "complexity_tool": "thop",
            "complexity_status": "unavailable",
            "macs": None,
            "flops_convention": "not_computed",
            "message": str(exc),
        }

    was_training = model.training
    model.eval()
    dummy = torch.randn(*input_size, device=device)
    try:
        macs, params = profile(model, inputs=(dummy,), verbose=False)
        return {
            "complexity_tool": "thop",
            "complexity_status": "ok",
            "macs": int(macs),
            "params_seen_by_tool": int(params),
            "flops_convention": "thop returns MACs; FLOPs are often approximated as 2x MACs",
            "estimated_flops_if_2x_macs": int(2 * macs),
        }
    except Exception as exc:  # pragma: no cover - model/operator dependent
        return {
            "complexity_tool": "thop",
            "complexity_status": "failed",
            "macs": None,
            "flops_convention": "not_computed",
            "message": str(exc),
        }
    finally:
        if was_training:
            model.train()


def save_json(data: Any, path: str | Path) -> Path:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    return path


def save_csv(rows: list[dict[str, Any]], path: str | Path) -> Path:
    path = Path(path)
    ensure_dir(path.parent)
    if not rows:
        raise ValueError("empty csv")
    fieldnames: list[str] = sorted({key for row in rows for key in row})
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path


def save_training_history(history: list[dict[str, Any]], path: str | Path) -> Path:
    return save_csv(history, path)


def plot_training_history(
    history: list[dict[str, Any]],
    title: str,
    path: str | Path,
) -> Path:
    """Save loss and accuracy curves from a per-epoch history table."""

    path = Path(path)
    ensure_dir(path.parent)
    epochs = [row["epoch"] for row in history]
    train_loss = [row.get("train_loss") for row in history]
    val_acc = [row.get("eval_accuracy") for row in history]

    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(epochs, train_loss, marker="o", label="Train loss", color="#1f77b4")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Training loss")
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(epochs, val_acc, marker="s", label="Eval accuracy", color="#2ca02c")
    ax2.set_ylabel("Eval accuracy")

    lines = ax1.get_lines() + ax2.get_lines()
    labels = [line.get_label() for line in lines]
    ax1.legend(lines, labels, loc="best")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


"""CIFAR-100 data loading and preprocessing helpers."""


from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from torchvision import datasets, transforms


CIFAR100_NUM_CLASSES = 100
CIFAR100_IMAGE_SIZE = 32
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD = (0.2675, 0.2565, 0.2761)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def build_cifar100_transforms(
    image_size: int,
    train: bool,
    mean: tuple[float, float, float] = CIFAR100_MEAN,
    std: tuple[float, float, float] = CIFAR100_STD,
    augment: bool = True,
) -> transforms.Compose:
    """Create transforms for CIFAR-100 at either native or resized resolution."""

    steps: list[object] = []
    if image_size != CIFAR100_IMAGE_SIZE:
        steps.append(transforms.Resize((image_size, image_size), antialias=True))
    if train and augment:
        if image_size == CIFAR100_IMAGE_SIZE:
            steps.extend(
                [
                    transforms.RandomCrop(CIFAR100_IMAGE_SIZE, padding=4),
                    transforms.RandomHorizontalFlip(),
                ]
            )
        else:
            steps.append(transforms.RandomHorizontalFlip())
    steps.extend([transforms.ToTensor(), transforms.Normalize(mean, std)])
    return transforms.Compose(steps)


def _limit_dataset(dataset: Dataset, max_items: int | None) -> Dataset:
    if max_items is None:
        return dataset
    return Subset(dataset, list(range(min(max_items, len(dataset)))))


def get_cifar100_dataloaders(
    data_root: str | Path,
    batch_size: int,
    image_size: int,
    mean: tuple[float, float, float] = CIFAR100_MEAN,
    std: tuple[float, float, float] = CIFAR100_STD,
    augment_train: bool = True,
    num_workers: int = 2,
    pin_memory: bool | None = None,
    validation_size: int = 0,
    seed: int = 42,
    max_train_items: int | None = None,
    max_test_items: int | None = None,
    download: bool = True,
) -> dict[str, DataLoader]:
    """Download CIFAR-100 and return train/test or train/val/test loaders."""

    data_root = Path(data_root)
    train_transform = build_cifar100_transforms(image_size, True, mean, std, augment_train)
    test_transform = build_cifar100_transforms(image_size, False, mean, std, False)

    train_dataset = datasets.CIFAR100(
        root=data_root,
        train=True,
        download=download,
        transform=train_transform,
    )
    test_dataset = datasets.CIFAR100(
        root=data_root,
        train=False,
        download=download,
        transform=test_transform,
    )

    if validation_size > 0:
        if validation_size >= len(train_dataset):
            raise ValueError("bad validation_size")
        train_size = len(train_dataset) - validation_size
        generator = torch.Generator().manual_seed(seed)
        train_dataset, val_dataset = random_split(
            train_dataset,
            [train_size, validation_size],
            generator=generator,
        )
    else:
        val_dataset = None

    train_dataset = _limit_dataset(train_dataset, max_train_items)
    test_dataset = _limit_dataset(test_dataset, max_test_items)
    if val_dataset is not None:
        val_dataset = _limit_dataset(val_dataset, max_test_items)

    if pin_memory is None:
        pin_memory = torch.cuda.is_available()

    loader_kwargs = {
        "batch_size": batch_size,
        "num_workers": num_workers,
        "pin_memory": pin_memory,
    }
    loaders = {
        "train": DataLoader(train_dataset, shuffle=True, **loader_kwargs),
        "test": DataLoader(test_dataset, shuffle=False, **loader_kwargs),
    }
    if val_dataset is not None:
        loaders["val"] = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
    return loaders


def validate_cifar100_labels(loader: DataLoader) -> tuple[int, int]:
    """Return min/max labels from one loader and assert CIFAR-100 range."""

    min_label = CIFAR100_NUM_CLASSES
    max_label = -1
    for _, labels in loader:
        min_label = min(min_label, int(labels.min().item()))
        max_label = max(max_label, int(labels.max().item()))
    if min_label < 0 or max_label >= CIFAR100_NUM_CLASSES:
        raise ValueError(f"bad labels: {min_label}-{max_label}")
    return min_label, max_label


"""Reusable supervised training and evaluation loops."""


import time
from pathlib import Path
from typing import Iterable

import torch
from torch import nn



def extract_logits(model_output):
    """Handle plain Tensor outputs and Hugging Face outputs with .logits."""

    if hasattr(model_output, "logits"):
        return model_output.logits
    if isinstance(model_output, tuple):
        return model_output[0]
    return model_output


def set_named_modules_eval(model: nn.Module, module_names: Iterable[str]) -> None:
    """Put selected top-level modules into eval mode during head-only training."""

    for name in module_names:
        module = getattr(model, name, None)
        if isinstance(module, nn.Module):
            module.eval()


def train_one_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device | str,
    max_batches: int | None = None,
    frozen_eval_modules: Iterable[str] = (),
) -> dict[str, float]:
    """Train for one epoch and return loss, accuracy, and elapsed seconds."""

    device = torch.device(device)
    model.train()
    set_named_modules_eval(model, frozen_eval_modules)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    cuda_synchronize_if_needed(device)
    start = time.perf_counter()

    for batch_idx, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = extract_logits(model(images))
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

    cuda_synchronize_if_needed(device)
    elapsed = time.perf_counter() - start
    if total_examples == 0:
        raise RuntimeError("no train batches")
    return {
        "train_loss": total_loss / total_examples,
        "train_accuracy": total_correct / total_examples,
        "epoch_time_seconds": elapsed,
    }


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    device: torch.device | str,
    max_batches: int | None = None,
) -> dict[str, float]:
    """Evaluate with gradients disabled."""

    device = torch.device(device)
    was_training = model.training
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for batch_idx, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = extract_logits(model(images))
        loss = criterion(logits, labels)
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size
    if was_training:
        model.train()
    if total_examples == 0:
        raise RuntimeError("no eval batches")
    return {
        "eval_loss": total_loss / total_examples,
        "eval_accuracy": total_correct / total_examples,
    }


def save_checkpoint(
    model: nn.Module,
    path: str | Path,
    metadata: dict | None = None,
) -> Path:
    """Save model weights plus serializable metadata."""

    path = Path(path)
    ensure_dir(path.parent)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "metadata": metadata or {},
        },
        path,
    )
    return path


def fit_classifier(
    model: nn.Module,
    train_loader,
    eval_loader,
    optimizer: torch.optim.Optimizer,
    device: torch.device | str,
    epochs: int,
    checkpoint_path: str | Path | None = None,
    max_train_batches: int | None = None,
    max_eval_batches: int | None = None,
    frozen_eval_modules: Iterable[str] = (),
) -> tuple[list[dict], dict]:
    """Train/evaluate a classifier and return per-epoch history plus summary."""

    criterion = nn.CrossEntropyLoss()
    history: list[dict] = []
    best_accuracy = -1.0
    total_start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
            max_batches=max_train_batches,
            frozen_eval_modules=frozen_eval_modules,
        )
        eval_metrics = evaluate(
            model,
            eval_loader,
            criterion,
            device,
            max_batches=max_eval_batches,
        )
        row = {"epoch": epoch, **train_metrics, **eval_metrics}
        history.append(row)
        if checkpoint_path is not None and eval_metrics["eval_accuracy"] >= best_accuracy:
            best_accuracy = eval_metrics["eval_accuracy"]
            save_checkpoint(
                model,
                checkpoint_path,
                {"best_epoch": epoch, "best_eval_accuracy": best_accuracy},
            )

    total_time = time.perf_counter() - total_start
    summary = {
        "total_training_time_seconds": total_time,
        "mean_epoch_time_seconds": sum(r["epoch_time_seconds"] for r in history) / len(history),
        "final_eval_accuracy": history[-1]["eval_accuracy"],
        "final_eval_loss": history[-1]["eval_loss"],
    }
    return history, summary


def verify_optimizer_parameters(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
) -> tuple[int, int]:
    """Return trainable parameter count and optimizer parameter count."""

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    optimized = 0
    seen: set[int] = set()
    for group in optimizer.param_groups:
        for param in group["params"]:
            if id(param) not in seen:
                optimized += param.numel()
                seen.add(id(param))
    if trainable != optimized:
        raise ValueError("bad optimizer params")
    return trainable, optimized


Define the scratch Vision Transformer architecture and the four required ViT configurations.


In [3]:
"""Vision Transformer from scratch for CIFAR-100."""


from dataclasses import asdict, dataclass

import torch
from torch import nn


@dataclass(frozen=True)
class ViTConfig:
    name: str
    image_size: int = 32
    patch_size: int = 4
    in_channels: int = 3
    num_classes: int = 100
    embed_dim: int = 256
    depth: int = 4
    num_heads: int = 4
    dropout: float = 0.1

    @property
    def mlp_hidden_dim(self) -> int:
        return 4 * self.embed_dim

    def validate(self) -> None:
        if self.image_size % self.patch_size != 0:
            raise ValueError("bad patch_size")
        if self.embed_dim % self.num_heads != 0:
            raise ValueError("bad heads")

    def as_dict(self) -> dict:
        data = asdict(self)
        data["mlp_hidden_dim"] = self.mlp_hidden_dim
        return data


VIT_CONFIGS: dict[str, ViTConfig] = {
    "vit_p4_d256_l4_h4": ViTConfig(
        name="vit_p4_d256_l4_h4", patch_size=4, embed_dim=256, depth=4, num_heads=4
    ),
    "vit_p4_d512_l8_h8": ViTConfig(
        name="vit_p4_d512_l8_h8", patch_size=4, embed_dim=512, depth=8, num_heads=8
    ),
    "vit_p8_d256_l4_h4": ViTConfig(
        name="vit_p8_d256_l4_h4", patch_size=8, embed_dim=256, depth=4, num_heads=4
    ),
    "vit_p8_d512_l8_h8": ViTConfig(
        name="vit_p8_d512_l8_h8", patch_size=8, embed_dim=512, depth=8, num_heads=8
    ),
}


class PatchEmbedding(nn.Module):
    """Image to patch-token projection using a Conv2d patchifier."""

    def __init__(self, image_size: int, patch_size: int, in_channels: int, embed_dim: int):
        super().__init__()
        if image_size % patch_size != 0:
            raise ValueError("bad patch_size")
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x


class MultiHeadSelfAttention(nn.Module):
    """Explicit multi-head self-attention with qkv projections."""

    def __init__(self, embed_dim: int, num_heads: int, dropout: float):
        super().__init__()
        if embed_dim % num_heads != 0:
            raise ValueError("bad heads")
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim**-0.5
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz, seq_len, dim = x.shape
        qkv = self.qkv(x)
        qkv = qkv.reshape(bsz, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        out = attn @ v
        out = out.transpose(1, 2).reshape(bsz, seq_len, dim)
        out = self.proj(out)
        return self.proj_drop(out)


class TransformerEncoderBlock(nn.Module):
    """Pre-norm Transformer encoder block."""

    def __init__(self, embed_dim: int, num_heads: int, mlp_hidden_dim: int, dropout: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class VisionTransformer(nn.Module):
    """Scratch Vision Transformer classifier."""

    def __init__(self, config: ViTConfig):
        super().__init__()
        config.validate()
        self.config = config
        self.patch_embed = PatchEmbedding(
            config.image_size,
            config.patch_size,
            config.in_channels,
            config.embed_dim,
        )
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, config.embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, config.embed_dim))
        self.pos_drop = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(
            *[
                TransformerEncoderBlock(
                    config.embed_dim,
                    config.num_heads,
                    config.mlp_hidden_dim,
                    config.dropout,
                )
                for _ in range(config.depth)
            ]
        )
        self.norm = nn.LayerNorm(config.embed_dim)
        self.head = nn.Linear(config.embed_dim, config.num_classes)
        self._init_weights()

    def _init_weights(self) -> None:
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz = x.size(0)
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(bsz, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        if x.size(1) != self.pos_embed.size(1):
            raise ValueError("bad pos_embed")
        x = self.pos_drop(x + self.pos_embed)
        x = self.blocks(x)
        x = self.norm(x)
        return self.head(x[:, 0])


def build_vit(config_name: str) -> VisionTransformer:
    return VisionTransformer(VIT_CONFIGS[config_name])


def manual_vit_parameter_breakdown(config: ViTConfig) -> dict[str, int]:
    """Manual parameter calculation for the scratch ViT configuration."""

    config.validate()
    patches_per_side = config.image_size // config.patch_size
    num_patches = patches_per_side**2
    d = config.embed_dim
    mlp = config.mlp_hidden_dim
    patch_embedding = d * config.in_channels * config.patch_size * config.patch_size + d
    class_token = d
    positional_embedding = (num_patches + 1) * d
    attention_per_block = (d * 3 * d + 3 * d) + (d * d + d)
    mlp_per_block = (d * mlp + mlp) + (mlp * d + d)
    layer_norm_per_block = 4 * d
    block_total = attention_per_block + mlp_per_block + layer_norm_per_block
    final_norm = 2 * d
    classification_head = d * config.num_classes + config.num_classes
    total = (
        patch_embedding
        + class_token
        + positional_embedding
        + config.depth * block_total
        + final_norm
        + classification_head
    )
    return {
        "patch_embedding": patch_embedding,
        "class_token": class_token,
        "positional_embedding": positional_embedding,
        "attention_projections_all_blocks": config.depth * attention_per_block,
        "mlp_layers_all_blocks": config.depth * mlp_per_block,
        "layer_norms_all_blocks": config.depth * layer_norm_per_block,
        "final_layer_norm": final_norm,
        "classification_head": classification_head,
        "manual_total": total,
    }


def verify_vit_shapes(config: ViTConfig, batch_size: int = 2) -> tuple[torch.Size, int]:
    """Run a synthetic forward pass and return output shape plus token length."""

    model = VisionTransformer(config)
    x = torch.randn(batch_size, config.in_channels, config.image_size, config.image_size)
    logits = model(x)
    expected = (batch_size, config.num_classes)
    if tuple(logits.shape) != expected:
        raise AssertionError("bad logits")
    return logits.shape, model.pos_embed.size(1)


Import the remaining Problem 1 tools, set the seed, select the device, and define the required hyperparameters.


In [4]:
import json

import torch
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18

WORK_DIR, DATA_ROOT, RESULT_DIR = project_paths("problem1")

SEED = 42
set_seed(SEED)
device_info = select_device()
device = device_info.device
print("Selected device:", device_info.as_dict())

EPOCHS = 10
BATCH_SIZE = 64
LEARNING_RATE = 0.001
ensure_dir(RESULT_DIR / "checkpoints")
ensure_dir(RESULT_DIR / "plots")


Mounted at /content/drive
Selected device: {'device': 'cuda', 'name': 'Tesla T4', 'cuda_available': True, 'cuda_device_count': 1}


PosixPath('/content/drive/MyDrive/homework_5/results/problem1/plots')

Load CIFAR-100 with 32x32 preprocessing for ViT and 224x224 ImageNet preprocessing for ResNet-18.


In [5]:
vit_loaders = get_cifar100_dataloaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    image_size=32,
    mean=CIFAR100_MEAN,
    std=CIFAR100_STD,
    augment_train=True,
    num_workers=2,
    seed=SEED,
    download=True,
)
resnet_loaders = get_cifar100_dataloaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    image_size=224,
    mean=IMAGENET_MEAN,
    std=IMAGENET_STD,
    augment_train=True,
    num_workers=2,
    seed=SEED,
    download=True,
)

print("ViT test label range:", validate_cifar100_labels(vit_loaders["test"]))
images, labels = next(iter(vit_loaders["train"]))
print("ViT training batch:", images.shape, labels.min().item(), labels.max().item())


ViT test label range: (0, 99)
ViT training batch: torch.Size([64, 3, 32, 32]) 0 98


Define the ResNet-18 builder and a reusable experiment runner for training, timing, checkpointing, and exporting results.


In [6]:
def build_resnet18_for_cifar100():
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 100)
    return model, str(weights)


def execute_model_experiment(model_name, model, loaders, input_size, config, pretrained, frozen_backbone):
    model = model.to(device)
    dummy = torch.randn(*input_size, device=device)
    logits = model(dummy)
    if hasattr(logits, "logits"):
        logits = logits.logits
    assert tuple(logits.shape) == (input_size[0], 100), "bad logits"

    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
    verify_optimizer_parameters(model, optimizer)

    total_params, trainable_params = count_parameters(model)
    complexity = estimate_macs_thop(model, input_size, device)
    checkpoint_path = RESULT_DIR / "checkpoints" / f"{model_name}.pt"

    history, fit_summary = fit_classifier(
        model=model,
        train_loader=loaders["train"],
        eval_loader=loaders["test"],
        optimizer=optimizer,
        device=device,
        epochs=EPOCHS,
        checkpoint_path=checkpoint_path,
    )

    history_path = RESULT_DIR / f"training_history_{model_name}.csv"
    plot_path = RESULT_DIR / "plots" / f"{model_name}_training_curves.png"
    save_training_history(history, history_path)
    plot_training_history(history, model_name, plot_path)

    row = {
        "model_name": model_name,
        "configuration": json.dumps(config),
        "total_parameters": total_params,
        "trainable_parameters": trainable_params,
        "model_size_mb": model_size_megabytes(model),
        "macs_per_forward": complexity.get("macs"),
        "estimated_flops_if_2x_macs": complexity.get("estimated_flops_if_2x_macs"),
        "complexity_tool": complexity.get("complexity_tool"),
        "complexity_status": complexity.get("complexity_status"),
        "flops_convention": complexity.get("flops_convention"),
        "total_training_time_seconds": fit_summary["total_training_time_seconds"],
        "mean_epoch_time_seconds": fit_summary["mean_epoch_time_seconds"],
        "test_accuracy": fit_summary["final_eval_accuracy"],
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "optimizer": "Adam",
        "random_seed": SEED,
        "device": str(device),
        "device_name": device_info.name,
        "pretrained_weights_used": pretrained,
        "frozen_backbone": frozen_backbone,
        "checkpoint_path": str(checkpoint_path),
        "history_path": str(history_path),
        "plot_path": str(plot_path),
    }
    return row, {"history": history, "summary": fit_summary, "complexity": complexity}


Train all required scratch ViT configurations and then train the pretrained ResNet-18 baseline.


In [7]:
all_results = []
detail_results = {}

for name, config in VIT_CONFIGS.items():
    print(f"Running {name}")
    model = VisionTransformer(config)
    manual = manual_vit_parameter_breakdown(config)
    programmatic_total, _ = count_parameters(model)
    manual["programmatic_total"] = programmatic_total
    manual["matches_programmatic_total"] = manual["manual_total"] == programmatic_total
    print("Manual parameter check:", manual)

    row, details = execute_model_experiment(
        model_name=name,
        model=model,
        loaders=vit_loaders,
        input_size=(1, 3, 32, 32),
        config={**config.as_dict(), "manual_parameter_breakdown": manual},
        pretrained=False,
        frozen_backbone=False,
    )
    all_results.append(row)
    detail_results[name] = details

print("Running pretrained ResNet-18 baseline")
resnet_model, resnet_weights_name = build_resnet18_for_cifar100()
row, details = execute_model_experiment(
    model_name="resnet18_pretrained_imagenet",
    model=resnet_model,
    loaders=resnet_loaders,
    input_size=(1, 3, 224, 224),
    config={
        "architecture": "torchvision.models.resnet18",
        "weights": resnet_weights_name,
        "input_size": 224,
        "normalization": "ImageNet",
        "classifier_outputs": 100,
    },
    pretrained=True,
    frozen_backbone=False,
)
all_results.append(row)
detail_results["resnet18_pretrained_imagenet"] = details


Running vit_p4_d256_l4_h4
Manual parameter check: {'patch_embedding': 12544, 'class_token': 256, 'positional_embedding': 16640, 'attention_projections_all_blocks': 1052672, 'mlp_layers_all_blocks': 2102272, 'layer_norms_all_blocks': 4096, 'final_layer_norm': 512, 'classification_head': 25700, 'manual_total': 3214692, 'programmatic_total': 3214692, 'matches_programmatic_total': True}
Running vit_p4_d512_l8_h8
Manual parameter check: {'patch_embedding': 25088, 'class_token': 512, 'positional_embedding': 33280, 'attention_projections_all_blocks': 8404992, 'mlp_layers_all_blocks': 16797696, 'layer_norms_all_blocks': 16384, 'final_layer_norm': 1024, 'classification_head': 51300, 'manual_total': 25330276, 'programmatic_total': 25330276, 'matches_programmatic_total': True}
Running vit_p8_d256_l4_h4
Manual parameter check: {'patch_embedding': 49408, 'class_token': 256, 'positional_embedding': 4352, 'attention_projections_all_blocks': 1052672, 'mlp_layers_all_blocks': 2102272, 'layer_norms_all_

100%|██████████| 44.7M/44.7M [00:00<00:00, 177MB/s]


Save the Problem 1 CSV and JSON outputs.


In [8]:
save_csv(all_results, RESULT_DIR / "problem1_results.csv")
save_json({"results": all_results, "details": detail_results}, RESULT_DIR / "problem1_results.json")
print("Saved Problem 1 artifacts to:", RESULT_DIR)
all_results


Saved Problem 1 artifacts to: /content/drive/MyDrive/homework_5/results/problem1


[{'model_name': 'vit_p4_d256_l4_h4',
  'configuration': '{"name": "vit_p4_d256_l4_h4", "image_size": 32, "patch_size": 4, "in_channels": 3, "num_classes": 100, "embed_dim": 256, "depth": 4, "num_heads": 4, "dropout": 0.1, "mlp_hidden_dim": 1024, "manual_parameter_breakdown": {"patch_embedding": 12544, "class_token": 256, "positional_embedding": 16640, "attention_projections_all_blocks": 1052672, "mlp_layers_all_blocks": 2102272, "layer_norms_all_blocks": 4096, "final_layer_norm": 512, "classification_head": 25700, "manual_total": 3214692, "programmatic_total": 3214692, "matches_programmatic_total": true}}',
  'total_parameters': 3214692,
  'trainable_parameters': 3214692,
  'model_size_mb': 12.263076782226562,
  'macs_per_forward': 205883392,
  'estimated_flops_if_2x_macs': 411766784,
  'complexity_tool': 'thop',
  'complexity_status': 'ok',
  'flops_convention': 'thop returns MACs; FLOPs are often approximated as 2x MACs',
  'total_training_time_seconds': 365.1804628680002,
  'mean_ep

Create a downloadable zip file and list the saved Problem 1 artifacts.


In [9]:
import shutil

for path in sorted(RESULT_DIR.rglob("*")):
    if path.is_file():
        print(path)

zip_path = shutil.make_archive(str(RESULT_DIR), "zip", RESULT_DIR)
print("Results zip:", zip_path)


/content/drive/MyDrive/homework_5/results/problem1/checkpoints/resnet18_pretrained_imagenet.pt
/content/drive/MyDrive/homework_5/results/problem1/checkpoints/vit_p4_d256_l4_h4.pt
/content/drive/MyDrive/homework_5/results/problem1/checkpoints/vit_p4_d512_l8_h8.pt
/content/drive/MyDrive/homework_5/results/problem1/checkpoints/vit_p8_d256_l4_h4.pt
/content/drive/MyDrive/homework_5/results/problem1/checkpoints/vit_p8_d512_l8_h8.pt
/content/drive/MyDrive/homework_5/results/problem1/plots/resnet18_pretrained_imagenet_training_curves.png
/content/drive/MyDrive/homework_5/results/problem1/plots/vit_p4_d256_l4_h4_training_curves.png
/content/drive/MyDrive/homework_5/results/problem1/plots/vit_p4_d512_l8_h8_training_curves.png
/content/drive/MyDrive/homework_5/results/problem1/plots/vit_p8_d256_l4_h4_training_curves.png
/content/drive/MyDrive/homework_5/results/problem1/plots/vit_p8_d512_l8_h8_training_curves.png
/content/drive/MyDrive/homework_5/results/problem1/problem1_results.csv
/content/dr